# Atlas Resampling & ROI Time Series Extraction

This notebook documents the first step of the pipeline in detail: aligning the AAL-116
atlas to fMRIPrep's output space, then extracting one BOLD time series per brain region
for every subject. A plain-script equivalent (no explanatory markdown) lives at
`src/01_extract_roi_timeseries.py`.

Data: [OpenNeuro ds005892](https://openneuro.org/datasets/ds005892/versions/1.0.0)


In [1]:
import sys, os
sys.path.append("..")  # repo root, so config.py is importable from notebooks/

import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

from config import SUBJECTS, FMRIPREP_DIR, AAL_ATLAS_PATH, OUTPUT_DIR, RESAMPLED_ATLAS_PATH

print("Number of subjects:", len(SUBJECTS))


Number of subjects: 55


## Check that every subject shares the same fMRI grid

If shape and affine are identical across subjects, the atlas only needs to be resampled **once** and reused for everyone.

In [2]:
for sub in SUBJECTS:
    fmri_path = os.path.join(
        FMRIPREP_DIR, sub, "func",
        f"{sub}_task-rest_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
    )
    img = nib.load(fmri_path)
    print(sub, img.shape, "\n", img.affine, "\n")


sub-MJF001 (50, 62, 48, 200) 
 [[   3.125    0.       0.     -77.25 ]
 [   0.       3.125    0.    -113.25 ]
 [   0.       0.       3.5    -78.   ]
 [   0.       0.       0.       1.   ]] 

sub-MJF002 (50, 62, 48, 200) 
 [[   3.125    0.       0.     -77.25 ]
 [   0.       3.125    0.    -113.25 ]
 [   0.       0.       3.5    -78.   ]
 [   0.       0.       0.       1.   ]] 

sub-MJF003 (50, 62, 48, 200) 
 [[   3.125    0.       0.     -77.25 ]
 [   0.       3.125    0.    -113.25 ]
 [   0.       0.       3.5    -78.   ]
 [   0.       0.       0.       1.   ]] 

sub-MJF006 (50, 62, 48, 200) 
 [[   3.125    0.       0.     -77.25 ]
 [   0.       3.125    0.    -113.25 ]
 [   0.       0.       3.5    -78.   ]
 [   0.       0.       0.       1.   ]] 

sub-MJF007 (50, 62, 48, 200) 
 [[   3.125    0.       0.     -77.25 ]
 [   0.       3.125    0.    -113.25 ]
 [   0.       0.       3.5    -78.   ]
 [   0.       0.       0.       1.   ]] 

sub-MJF008 (50, 62, 48, 200) 
 [[   3.125    0.   

All subjects have the same fMRI shape and affine matrix. Therefore, the atlas only needs
to be resampled once to the common fMRI space.

This resampled atlas can then be reused for ROI extraction for all subjects.

# Load one fMRI (reference) and atlas

In [3]:
reference_subject = SUBJECTS[0]
fmri_path = os.path.join(
    FMRIPREP_DIR, reference_subject, "func",
    f"{reference_subject}_task-rest_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
)
atlas_path = AAL_ATLAS_PATH

fmri_img = nib.load(fmri_path)
atlas_img = nib.load(atlas_path)

fmri_data = fmri_img.get_fdata()
atlas_data = atlas_img.get_fdata()

print("fMRI shape:", fmri_data.shape)
print("Atlas shape:", atlas_data.shape)
print("fMRI affine:\n", fmri_img.affine)
print("Atlas affine:\n", atlas_img.affine)


fMRI shape: (50, 62, 48, 200)
Atlas shape: (91, 109, 91)
fMRI affine:
 [[   3.125    0.       0.     -77.25 ]
 [   0.       3.125    0.    -113.25 ]
 [   0.       0.       3.5    -78.   ]
 [   0.       0.       0.       1.   ]]
Atlas affine:
 [[  -2.    0.    0.   90.]
 [   0.    2.    0. -126.]
 [   0.    0.    2.  -72.]
 [   0.    0.    0.    1.]]


## Note:
Affine matrix:

$$
\begin{bmatrix}
a & 0 & 0 & x \\\\
0 & b & 0 & y \\\\
0 & 0 & c & z \\\\
0 & 0 & 0 & 1
\end{bmatrix}
$$

Where:
- $a, b, c$ = voxel sizes
- $x, y, z$ = translation (origin)

# fMRI and Atlas Shape / Affine Interpretation

### 🔹 fMRI Data

**Shape:** (50 ; 62 ; 48 ; 200)

This means:

* (50 × 62 × 48) are the spatial dimensions
* (200) is the number of time points

So, the fMRI image is a **4D dataset**, where each voxel contains a BOLD signal measured across time.

### 🔹 fMRI Affine Matrix

\begin{bmatrix}
3.125 & 0 & 0 & -77.25 \\\\
0 & 3.125 & 0 & -113.25 \\\\
0 & 0 & 3.5 & -78 \\\\
0 & 0 & 0 & 1
\end{bmatrix}

#### Interpretation:

**Voxel size:**

* X direction = (3.125) mm
* Y direction = (3.125) mm
* Z direction = (3.5) mm

This indicates the spatial resolution of the fMRI voxels.

**Origin (translation):**

* (x = -77.25)
* (y = -113.25)
* (z = -78)

This gives the position of voxel (0,0,0) in real-world brain coordinates.

**Orientation:**
Since the diagonal values are positive, the voxel axes are not flipped.

---

### 🔹 Atlas Data

**Shape:** (91 ; 109 ; 91)

This means the atlas is a **3D label image**, where each voxel contains a region index corresponding to a brain area.

### 🔹 Atlas Affine Matrix

\begin{bmatrix}
-2 & 0 & 0 & 90 \\\\
0 & 2 & 0 & -126 \\\\
0 & 0 & 2 & -72 \\\\
0 & 0 & 0 & 1
\end{bmatrix}

#### Interpretation:

**Voxel size:**

* X direction = (2) mm
* Y direction = (2) mm
* Z direction = (2) mm

So the atlas has a different spatial resolution from the fMRI image.

**Orientation:**
The value (-2) on the X-axis means the X direction is flipped.
This usually corresponds to a left-right inversion in voxel indexing, which is common in atlas files.

**Origin (translation):**

* (x = 90)
* (y = -126)
* (z = -72)

This shows that the atlas origin is also different from the fMRI origin.

---
### Key Differences Between fMRI and Atlas

| Property    | fMRI                   | Atlas              |
| ----------- | ---------------------- | ------------------ |
| Shape       | (50, 62, 48, 200)      | (91, 109, 91)      |
| Resolution  | 3.125 × 3.125 × 3.5 mm | 2 × 2 × 2 mm        |
| Data Type   | 4D (spatial + time)    | 3D (region labels) |
| Orientation | Standard               | Flipped (X-axis)   |
| Origin      | (-77.25, -113.25, -78) | (90, -126, -72)     |
| Grid Size   | Coarser                | Finer               |

---

### Conclusion

The fMRI image and atlas do not match directly because they differ in:

* shape
* voxel size
* orientation
* origin

Therefore, the atlas must be **resampled to the fMRI space** before ROI extraction.

# Atlas Resampling
- All subjects have the same fMRI shape and affine matrix. Therefore, the atlas only needs to be resampled once to the common fMRI space.
- This resampled atlas can then be reused for ROI extraction for all subjects
- Before ROI time series extraction, the atlas must be aligned to the spatial grid of the preprocessed fMRI image

Although both the atlas and the fMRI data may be in MNI space, they often differ in:

* spatial resolution
* voxel size
* image shape
* affine matrix
* orientation

Because of these differences, the atlas cannot be applied directly to the fMRI image

---

### Objective

The goal of atlas resampling is to transform the atlas so that it matches the spatial dimensions and coordinate system of the fMRI image.

This ensures that each atlas label correctly overlaps the corresponding brain voxels in the functional image.

---

### Input Data

This step uses:

* a preprocessed fMRI image in MNI space
* an anatomical atlas containing labeled brain regions

For example:

* fMRI shape: ((50, 62, 48, 200))
* Atlas shape: ((91, 109, 91))

The first three dimensions of the fMRI image represent the spatial dimensions, while the fourth dimension represents time.

---

### Why Resampling is Required

The fMRI image and atlas may differ in several properties:

| Property    | fMRI                   | Atlas              |
| ----------- | ---------------------- | ------------------- |
| Shape       | (50, 62, 48, 200)      | (91, 109, 91)       |
| Resolution  | 3.125 × 3.125 × 3.5 mm | 2 × 2 × 2 mm         |
| Data Type   | 4D functional image    | 3D label image      |
| Orientation | Standard               | Flipped in X-axis    |
| Origin      | (-77.25, -113.25, -78) | (90, -126, -72)      |

Because of these differences, direct voxel-wise mapping between the atlas and the fMRI image would be incorrect.

---

### Principle of Resampling

Atlas resampling consists of transforming the atlas image to the same voxel grid as the fMRI image.

This includes matching:

* spatial shape
* affine matrix
* voxel coordinates

Since the atlas contains **discrete region labels**, the interpolation method must preserve integer values.

Therefore, **nearest-neighbor interpolation** is used.

---

### Why Nearest-Neighbor Interpolation?

In a brain atlas, each voxel contains a region label such as 1, 2, 3, etc.

Using linear interpolation would create non-integer values, which would corrupt the atlas labels.

Nearest-neighbor interpolation preserves the original label values and keeps the atlas anatomically valid.

In [4]:
atlas_resampled_img = resample_to_img(
    source_img=atlas_img,
    target_img=fmri_img,
    interpolation="nearest"
)

atlas_resampled_data = atlas_resampled_img.get_fdata()

print("fMRI spatial shape:", fmri_data.shape[:3])
print("Resampled atlas shape:", atlas_resampled_data.shape)

print("fMRI affine:\n", fmri_img.affine)
print("Resampled atlas affine:\n", atlas_resampled_img.affine)

labels = np.unique(atlas_resampled_data)
print("Number of labels:", len(labels))
print("Min label:", labels.min())
print("Max label:", labels.max())

# Remove background label (0)
region_ids = labels[labels != 0]
print("Number of brain regions:", len(region_ids))

if fmri_data.shape[:3] == atlas_resampled_data.shape:
    print("Atlas successfully aligned with fMRI")
else:
    print("Shape mismatch")

os.makedirs(OUTPUT_DIR, exist_ok=True)
nib.save(atlas_resampled_img, RESAMPLED_ATLAS_PATH)
print("Resampled atlas saved at:", RESAMPLED_ATLAS_PATH)


fMRI spatial shape: (50, 62, 48)
Resampled atlas shape: (50, 62, 48)
fMRI affine:
 [[   3.125    0.       0.     -77.25 ]
 [   0.       3.125    0.    -113.25 ]
 [   0.       0.       3.5    -78.   ]
 [   0.       0.       0.       1.   ]]
Resampled atlas affine:
 [[   3.125    0.       0.     -77.25 ]
 [   0.       3.125    0.    -113.25 ]
 [   0.       0.       3.5    -78.   ]
 [   0.       0.       0.       1.   ]]
Number of labels: 117
Min label: 0.0
Max label: 9170.0
Number of brain regions: 116
Atlas successfully aligned with fMRI
Resampled atlas saved at: C:/Users/XPRISTO/Desktop/PD_PROJECT\derivatives\AAL_resampled.nii.gz


# ROI Time Series Extraction
- After preprocessing the fMRI data and aligning the atlas to the fMRI space, the next step is to extract the ROI time series for each subject
- An ROI (Region of Interest) represents a specific anatomical brain region defined by the atlas.
- For each ROI, the mean BOLD signal is computed across all voxels belonging to that region at every time point.

---

### Objective

The goal of this step is to convert voxel-level fMRI data into region-level signals that can later be used to compute functional connectivity.

This reduces the dimensionality of the data and makes the analysis more interpretable.

---

### Input Data

For each subject, the ROI extraction step uses:

* a preprocessed 4D fMRI image in MNI space
* a resampled atlas image aligned to the fMRI grid

The fMRI data has shape:

$$(X, Y, Z, T)$$

where:

* (X, Y, Z) are spatial dimensions
* (T) is the number of time points

The atlas has shape:

$$(X, Y, Z)$$

and contains integer labels representing brain regions.

---

### Principle of ROI Extraction

Each atlas label corresponds to one anatomical brain region.

For a given region:

1. all voxels belonging to that region are selected
2. their BOLD signals are extracted across time
3. the mean signal is computed at each time point

This produces one representative time series for each ROI.

---

### Mathematical Representation

Let $V_r$ be the set of voxels belonging to region $r$.
If $S_v(t)$ is the BOLD signal of voxel $v$ at time $t$, then the ROI signal for region $r$ is:

$$
ROI_r(t) = \\frac{1}{|V_r|} \\sum_{v \\in V_r} S_v(t)
$$

Where:

* $ROI_r(t)$ is the average signal of region $r$ at time $t$
* $|V_r|$ is the number of voxels in region $r$
* $S_v(t)$ is the signal of voxel $v$ at time $t$

---

### Output

The result is a matrix of shape:

$$(T, R)$$

where:

* $T$ = number of time points
* $R$ = number of brain regions

Each:

* row corresponds to one time point
* column corresponds to one ROI

For example, with 200 time points and 116 atlas regions: $(200, 116)$

---

### Interpretation

Each column in the ROI matrix represents the average BOLD activity of a specific brain region over time.

These ROI time series summarize the temporal dynamics of the brain at the regional level and are used in the next step to compute functional connectivity.

In [5]:
atlas_img = nib.load(RESAMPLED_ATLAS_PATH)
atlas_data = atlas_img.get_fdata()

for sub in SUBJECTS:
    fmri_path = os.path.join(
        FMRIPREP_DIR, sub, "func",
        f"{sub}_task-rest_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz"
    )

    subject_folder = os.path.join(OUTPUT_DIR, sub)
    output_csv = os.path.join(subject_folder, f"{sub}_roi_time_series.csv")

    fmri_img = nib.load(fmri_path)
    fmri_data = fmri_img.get_fdata()

    roi_time_series = []

    for region in region_ids:
        mask = atlas_data == region
        region_voxels = fmri_data[mask]

        if region_voxels.shape[0] == 0:
            continue

        mean_signal = region_voxels.mean(axis=0)
        roi_time_series.append(mean_signal)

    roi_time_series = np.array(roi_time_series).T

    df = pd.DataFrame(
        roi_time_series,
        columns=[f"ROI_{int(r)}" for r in region_ids[:roi_time_series.shape[1]]]
    )

    os.makedirs(subject_folder, exist_ok=True)
    df.to_csv(output_csv, index=False)

    print(f"{sub}: saved {df.shape} to {output_csv}")


sub-MJF001: saved (200, 116) to C:/Users/XPRISTO/Desktop/PD_PROJECT\derivatives\sub-MJF001\sub-MJF001_roi_time_series.csv
sub-MJF002: saved (200, 116) to C:/Users/XPRISTO/Desktop/PD_PROJECT\derivatives\sub-MJF002\sub-MJF002_roi_time_series.csv
sub-MJF003: saved (200, 116) to C:/Users/XPRISTO/Desktop/PD_PROJECT\derivatives\sub-MJF003\sub-MJF003_roi_time_series.csv
sub-MJF006: saved (200, 116) to C:/Users/XPRISTO/Desktop/PD_PROJECT\derivatives\sub-MJF006\sub-MJF006_roi_time_series.csv
sub-MJF007: saved (200, 116) to C:/Users/XPRISTO/Desktop/PD_PROJECT\derivatives\sub-MJF007\sub-MJF007_roi_time_series.csv
sub-MJF008: saved (200, 116) to C:/Users/XPRISTO/Desktop/PD_PROJECT\derivatives\sub-MJF008\sub-MJF008_roi_time_series.csv
sub-MJF009: saved (200, 116) to C:/Users/XPRISTO/Desktop/PD_PROJECT\derivatives\sub-MJF009\sub-MJF009_roi_time_series.csv
sub-MJF010: saved (200, 116) to C:/Users/XPRISTO/Desktop/PD_PROJECT\derivatives\sub-MJF010\sub-MJF010_roi_time_series.csv
sub-MJF011: saved (200, 